# Day 23: Reading Text Files

**Today's question:** how does a program read data that lives in a **file** instead of inside the code?

Before running a cell marked 🤔, predict what it will show.

## 1. Why files?

So far all our data was typed into the code: `marks = [72, 85, 64]`. Real data arrives in files: logs,
exports, notes, survey results. Our program should be able to read them **without us retyping anything**.

This notebook sits next to a `data` folder:

```
day-23-reading-files/
├── demo.ipynb            ← we are here
└── data/
    ├── class_diary.txt
    └── attendance.txt
```

The path `"data/class_diary.txt"` means: *from where I am, go into `data`, then take `class_diary.txt`*.
That is a **relative path**: relative to the folder the program is running in.

## 2. `open()`, `with` and `.read()`

In [ ]:
with open("data/class_diary.txt", encoding="utf-8") as diary_file:
    text = diary_file.read()

print(text)

Reading the cell piece by piece:

| Part | Meaning |
|------|---------|
| `open("data/class_diary.txt", ...)` | connect to the file (like opening a book) |
| `encoding="utf-8"` | how the bytes on disk turn into letters (always write it) |
| `with ... as diary_file:` | give the open file a name, and **close it automatically** when the indented block ends |
| `diary_file.read()` | the whole file as **one string** |

🤔 **Predict:** what type is `text`? How many `"\n"` (newline) characters does it contain?

In [ ]:
print(type(text))
print("Characters:", len(text))
print("Newlines:", text.count("\n"))

A file is just text. Every line ends with an invisible newline character `\n` (Day 04).

### Why `encoding="utf-8"`?

The diary contains `Zoë` and `café`. Watch what happens if Python uses the wrong encoding, which is what many
Windows computers do by default when you leave `encoding` out:

In [ ]:
with open("data/class_diary.txt", encoding="cp1252") as diary_file:
    wrong_text = diary_file.read()

print(wrong_text)

`Zoë` became `ZoÃ«`. No error, just wrong letters. **Always pass `encoding="utf-8"`.**

## 3. Looping over the lines

A file can be used directly in a `for` loop: one line per turn.

🤔 **Predict:** what will the output look like?

In [ ]:
with open("data/class_diary.txt", encoding="utf-8") as diary_file:
    for line in diary_file:
        print(line)

Double-spaced! Each `line` still ends with `\n`, and `print()` adds another one.
Collecting the lines into a list makes the hidden characters visible:

In [ ]:
with open("data/class_diary.txt", encoding="utf-8") as diary_file:
    raw_lines = []
    for line in diary_file:
        raw_lines.append(line)

print(raw_lines[:3])

`.strip()` (Day 04) removes spaces **and** newlines from both ends. With a comprehension (Day 22) we get clean
lines in one step:

In [ ]:
with open("data/class_diary.txt", encoding="utf-8") as diary_file:
    lines = [line.strip() for line in diary_file]

print(lines[:3])
print("Lines in file:", len(lines))

Line numbers with `enumerate(..., start=1)`, and skipping blank lines with `continue` (Day 14):

In [ ]:
for number, line in enumerate(lines, start=1):
    if line == "":
        continue
    print(f"{number:>2}: {line[:45]}")

## 4. Lines with fields: `split(",")`

Each line of `attendance.txt` holds three pieces of information separated by commas:

```
2026-11-02,Asha,present
```

🤔 **Predict:** what does `split(",")` give for that line?

In [ ]:
sample = "2026-11-02,Asha,present"
print(sample.split(","))

date, name, status = sample.split(",")
print(name, "was", status, "on", date)

The same idea for every line in the file. Who was absent?

In [ ]:
with open("data/attendance.txt", encoding="utf-8") as log_file:
    for line in log_file:
        line = line.strip()
        if line == "":
            continue
        date, name, status = line.split(",")
        if status == "absent":
            print(f"{date}: {name} was absent")

The file has a **blank line** in the middle (someone pressed Enter twice). Without the `if line == "": continue`
check, `"".split(",")` gives `['']`, and unpacking one item into three names would crash. Real files are messy;
check before you split.

Counting with a dictionary (the Day 12 pattern):

In [ ]:
absences = {}
with open("data/attendance.txt", encoding="utf-8") as log_file:
    for line in log_file:
        line = line.strip()
        if line == "":
            continue
        date, name, status = line.split(",")
        if status == "absent":
            absences[name] = absences.get(name, 0) + 1

print(absences)

## 5. Where is Python looking? Relative paths

A relative path starts from the **working directory**:
- in a notebook: the folder the notebook is in;
- with `uv run script.py` in a terminal: the folder the terminal is in.

`..` means "one folder up". The course `datasets` folder is three levels above this notebook:

In [ ]:
with open("../../../datasets/README.md", encoding="utf-8") as readme_file:
    readme = readme_file.read()

print(readme.split("\n")[0])

## 6. Common mistakes (these cells fail **on purpose**)

In [ ]:
# Mistake 1: a typo in the file name
with open("data/atendance.txt", encoding="utf-8") as log_file:
    text = log_file.read()

In [ ]:
# Mistake 2: forgetting the folder. Python looks next to the notebook, not inside data/
with open("class_diary.txt", encoding="utf-8") as diary_file:
    text = diary_file.read()

`FileNotFoundError` means: *nothing exists at that path, starting from the working directory*. Check the spelling,
the folder and where you are running from. (On Day 25 we will learn to handle this error instead of crashing.)

In [ ]:
# Mistake 3: using the file after the with block has closed it
with open("data/class_diary.txt", encoding="utf-8") as diary_file:
    print("File is open")
for line in diary_file:
    print(line)

And one with **no error** but a surprising result. 🤔 **Predict** both lengths:

In [ ]:
with open("data/class_diary.txt", encoding="utf-8") as diary_file:
    first = diary_file.read()
    second = diary_file.read()

print(len(first), len(second))

Reading moves a "bookmark" through the file. After the first `.read()` the bookmark is at the end, so there is
nothing left to read. **Read once, store the result in a variable, reuse the variable.**

## 7. Real-world example: statistics for a text file

Two small functions: one reads, one analyses. Note the new trick: `.strip(".,!?():")` removes **those characters**
from both ends of a word, so `"line."` and `"line"` count as the same word.

In [ ]:
def read_lines(path):
    """Return the lines of a text file with spaces and newlines stripped."""
    with open(path, encoding="utf-8") as text_file:
        return [line.strip() for line in text_file]


def print_statistics(lines):
    """Print line, word and length statistics for a list of lines."""
    text_lines = [line for line in lines if line != ""]
    words = " ".join(text_lines).split()
    clean_words = [word.strip(".,!?():").lower() for word in words]

    print(f"Lines: {len(lines)} ({len(text_lines)} with text)")
    print(f"Words: {len(words)}")
    print(f"Average words per text line: {len(words) / len(text_lines):.1f}")
    print(f"Longest word: {max(clean_words, key=len)}")
    print(f"Times 'line' appears: {clean_words.count('line')}")

In [ ]:
diary_lines = read_lines("data/class_diary.txt")
print_statistics(diary_lines)

## Recap

1. `with open(path, encoding="utf-8") as f:` opens a file and closes it automatically at the end of the block.
2. `.read()` gives the whole file as one string; `for line in f:` gives one line at a time.
3. Every line ends with `\n`: use `.strip()`, and skip blank lines before splitting.
4. `line.split(",")` breaks a line into fields; unpack them into names.
5. Relative paths start from the working directory. `FileNotFoundError` = wrong name, wrong folder or wrong place.

**Next time:** writing files, so programs can **save** results and remember things after they stop.